# QVC Data Engineer - Technical Challenge: Daily Returns Dataset Transformation Pipeline 
<hr style="border:2px solid gray">

**Objective:** Develop a robust production-ready notebook to transform a raw one-day returns dataset into a clean, analytics-ready final dataset.

**Time Allotment:** 4-6 hours. We are looking for :
1. clean, modular code
2. strong PySpark fundamentals
3. sound data engineering judgement
4. clear handling of data quality issues
5. sensible optimisation and storage choices




## Setup and Introduction (Candidate Preamble)
---




Please fill out this section upon starting the notebook.

| Parameter | Response |
| :--- | :--- |
| **Candidate Name** | Bhanu Teja |
| **Time Spent (Approx.)** | ~5 hours |
| **Key Assumptions Made** | `parcellab_system_created_date` + `created_time` are assumed to represent the same instant (same format, but not verified as guaranteed-atomic in the source system) - combined into `parcellab_created_ts` with this caveat documented. `refreshed_date` and 27/1000 `activity_monitor_last_update` values are corrupted/truncated timestamp fragments (e.g. `"38:26.6"`) - left unparsed/NULL rather than force-cast. `id` looked like a unique per-record key but is **not** (905 distinct values across 1000 rows - it identifies the shipment, and one shipment can have multiple rows, one per returned `article_number`); a surrogate `_row_uid` was introduced and used for all joins/grouping instead. Nested objects inside `articles`/`custom_fields` (`customFields`, `priceDetails`, `tracking`, `outboundCustomFields`) are flattened one level only, not recursively expanded. |
| **Output Format Chosen** | Delta (`output/final/returns_tracking_delta` and `output/exceptions/returns_tracking_exceptions_delta`) |
| **Any Limitations in the Submitted Solution** | The `custom_fields` schema was derived from this single sample day and validated against it (0 parse failures), but has not been seen against other days' data. This sample file has zero organic exception rows (every row parses and matches exactly once), so the exception-handling and repair logic (Part 3) is proven via a small synthetic bad-row harness rather than real failing rows. `refreshed_date` and the 27 corrupted `activity_monitor_last_update` values are left as NULL/raw rather than repaired.



### Business Context

You are provided with a sample CSV file representing one day of **returns** data.
The sample file contains:

1. Standard scalar columns
2. an **articles** column containing a list of JSON objects
3. a **custom_fields** column containing a nested JSON object

Your task is to transform this file into a stable, analytics-ready dataset that could realistically be used as part of a daily processing pipeline.
Structure the notebook in a way that would be reasonable for reuse in a production-style setting.
Please keep cluster efficiency in mind and make sensible optimisation choices where appropriate.

**Note:** 
The provided file is only a one-day sample. Your solution should be written as though it will be used on future daily files with similar structure and possible edge cases.




### Part 0: Pipeline Setup

Configuration, schemas/constants, and the reusable transform/validation/repair
functions used throughout this notebook live here. Each numbered Part below then
just calls into these functions and prints/asserts that task's specific evidence -
see the function bodies here for the actual implementation.


#### Configuration

In [ ]:
# ==== Configuration ====
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType
from delta.tables import DeltaTable

# Databricks provides the `spark` session and Delta support. Do not create a local
# SparkSession or install Delta packages in this notebook.
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")
spark.sparkContext.setLogLevel("WARN")
print("spark version", spark.version)

# This workspace uses the legacy Hive metastore. Upload the CSV through the UI into
# the input table below, then run this notebook. All outputs are managed Delta tables.
dbutils.widgets.text(
    "input_table", "hive_metastore.default.daily_returns_trackings_raw", "Input table"
)
dbutils.widgets.text(
    "output_schema", "hive_metastore.default", "Output schema"
)
INPUT_TABLE = dbutils.widgets.get("input_table").strip()
OUTPUT_SCHEMA = dbutils.widgets.get("output_schema").strip()
assert INPUT_TABLE and OUTPUT_SCHEMA, "Set both the input_table and output_schema widgets."

# Run identity, for idempotent MERGE writes and the run-audit table (see the
# Output & Audit cell). Generated once per notebook run.
PIPELINE_RUN_ID = str(uuid.uuid4())
PIPELINE_STARTED_AT = datetime.utcnow()
print("pipeline_run_id:", PIPELINE_RUN_ID)
print("pipeline_started_at (UTC):", PIPELINE_STARTED_AT)

# Best-effort source identity signal. Ingestion reads spark.table(INPUT_TABLE), not
# a file, so there is no file path/mtime/checksum available the way there would be
# with a file-based read (input_file_name(), a checksum column, etc.) - that gap is
# a documented limitation, not faked. If INPUT_TABLE happens to be Delta-backed
# (plausible for a Databricks CSV-upload-to-table flow), its own commit history
# gives a real, non-fabricated version signal instead.
try:
    _input_history = DeltaTable.forName(spark, INPUT_TABLE).history(1).collect()
    SOURCE_TABLE_VERSION = str(_input_history[0]["version"]) if _input_history else None
except Exception:
    SOURCE_TABLE_VERSION = None
print("source_table_version (best-effort, None if input table isn't Delta-backed):", SOURCE_TABLE_VERSION)


#### Schemas & Constants

In [ ]:
# ==== Schemas & Constants ====
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, IntegerType, DoubleType,
    ArrayType, MapType,
)

BOOL_COLS = [
    "is_returns_portal", "return_shipment", "is_cancelled", "is_complete",
    "is_branch_delivery", "cash_on_delivery", "is_transport", "is_doorstep_delivery",
    "has_pod_identifier", "has_pod_signature", "is_contacted", "is_contacted_and_bounce",
    "invalid", "forgotten", "dispatch_delayed", "network_delayed", "failed_attempt",
    "sla_exceeded", "customer_promise_exceeded", "returned_to_sender",
]

STRING_PRESERVE_COLS = ["id", "customer_no", "order_no", "tracking_number", "delivery_no", "xid"]

DATE_TIME_PAIRS = [
    ("order_date", "order_time", "order_ts"),
    ("pickup_scheduled_date", "pickup_scheduled_time", "pickup_scheduled_ts"),
    ("inbound_scan_date", "inbound_scan_time", "inbound_scan_ts"),
    ("qualified_delivery_attempt_date", "qualified_delivery_attempt_time", "qualified_delivery_attempt_ts"),
    ("delivery_date", "delivery_time", "delivery_ts"),
    ("announced_dispatch_date", "announced_dispatch_time", "announced_dispatch_ts"),
]

DATE_ONLY_COLS = {
    "cancelled_date": "cancelled_dt",
    "promise_date": "promise_dt",
    "return_to_sender_date": "return_to_sender_dt",
    "claimed_date": "claimed_dt",
}

SINGLE_TS_COLS = {
    "updated": "updated_ts",
    "activity_monitor_last_update": "activity_monitor_last_update_ts",
}

N_DAYS_COLS = [
    "delta_between_announced_and_actual_dispatch",
    "n_days_edi_transmission_until_final_delivery",
    "n_days_courier_inbound_until_first_attempt",
    "n_days_edi_transmission_until_first_attempt",
    "n_days_edi_transmission_until_courier_inbound",
    "n_days_order_until_first_attempt",
    "n_days_order_until_courier_inbound",
    "n_days_failed_until_delivered",
    "n_days_edi_transmission_customer_collected",
]

articles_item_schema = StructType([
    StructField("articleNo", StringType()),
    StructField("productId", StringType()),
    StructField("articleName", StringType()),
    StructField("articleUrl", StringType()),
    StructField("articleImageUrl", StringType()),
    StructField("articleBrand", StringType()),
    StructField("price", DoubleType()),
    StructField("articlePrice", DoubleType()),
    StructField("tax", DoubleType()),
    StructField("priceDetails", StructType([
        StructField("pricePaid", DoubleType()),
        StructField("pricePaidNoTax", DoubleType()),
    ])),
    StructField("size", StringType()),
    StructField("sizeCode", StringType()),
    StructField("color", StringType()),
    StructField("quantity", IntegerType()),
    StructField("returnReason", StringType()),
    StructField("returnReasonPath", StringType()),
    StructField("prettyReturnReason", StringType()),
    StructField("problemDescription", StringType()),
    StructField("compensationMethod", StringType()),
    StructField("returnImages", ArrayType(StringType())),
    StructField("lineNumbers", StringType()),
    StructField("itemId", StringType()),
    StructField("sku", StringType()),
    StructField("barcode", StringType()),
    StructField("prettyProductId", StringType()),
    StructField("season", StringType()),
    StructField("consignmentId", StringType()),
    StructField("weightInGrams", IntegerType()),
    StructField("siteId", StringType()),
    StructField("category", StringType()),
    StructField("articleCategory", StringType()),
    StructField("productType", StringType()),
    StructField("keepArticle", StringType()),
    StructField("invoiceNo", StringType()),
    StructField("invoiceLineItemId", StringType()),
    StructField("lineItemId", StringType()),
    StructField("fulfillmentOriginLocationId", StringType()),
    StructField("itemFulfillmentLineUniqueKey", StringType()),
    StructField("condition", StringType()),
    StructField("prettyCondition", StringType()),
    StructField("countryCodeOfOrigin", StringType()),
    StructField("countryOfManufacture", StringType()),
    StructField("harmonizedSystemCode", StringType()),
    StructField("requiresExtraMaterial", StringType()),
    StructField("returnImagesUrlList", StringType()),
    StructField("shopifyFulfillmentLineItems", StringType()),
    StructField("shopifyLineItemId", StringType()),
    StructField("customFields", StructType([
        StructField("gross_weight", StringType()),
        StructField("net_weight", StringType()),
        StructField("height", StringType()),
        StructField("width", StringType()),
        StructField("length", StringType()),
    ])),
    StructField("tracking", StructType([
        StructField("client", StringType()),
        StructField("courier", StringType()),
        StructField("deliveryNo", StringType()),
        StructField("tracking_number", StringType()),
        StructField("warehouse", StringType()),
        StructField("consignmentNo", StringType()),
        StructField("market", StringType()),
        StructField("transportNo", StringType()),
    ])),
])
articles_schema = ArrayType(articles_item_schema)
item_fields = [f.name for f in articles_item_schema.fields]

file_ref_schema = StructType([
    StructField("url", StringType()),
    StructField("bucket", StringType()),
    StructField("objectKey", StringType()),
    StructField("type", StringType()),
])

custom_fields_schema = StructType([
    StructField("isAdditionalLabel", BooleanType()),
    StructField("isManuallyAddedLabel", BooleanType()),
    StructField("originalCourier", StringType()),
    StructField("paymentMethod", StringType()),
    StructField("printLabel", file_ref_schema),
    StructField("backupPrintLabel", StringType()),
    StructField("barCode", file_ref_schema),
    StructField("customsDocument", StringType()),
    StructField("packingSlip", StringType()),
    StructField("claimsDocument", StringType()),
    StructField("commercialInvoice", StringType()),
    StructField("freeReturnLabel", BooleanType()),
    StructField("isWarranty", BooleanType()),
    StructField("isCourierPickup", BooleanType()),
    StructField("identityPin", StringType()),
    StructField("deliveryPointId", StringType()),
    StructField("groupCode", StringType()),
    StructField("pickupDateRange", StringType()),
    StructField("selectedLocation", StringType()),
    StructField("qualityCheckExpected", BooleanType()),
    StructField("shopifyReturnData", StringType()),
    StructField("shopifyCustomerTags", StringType()),
    StructField("shopifyOrderTags", ArrayType(StringType())),
    StructField("refundMethod", StringType()),
    StructField("labelCost", StringType()),
    StructField("changedAddress", BooleanType()),
    StructField("customerService", BooleanType()),
    StructField("secondHandCustomer", BooleanType()),
    StructField("outboundCustomFields", StructType([
        StructField("PaymentType", StringType()),
        StructField("ShipType", StringType()),
        StructField("ParcelShopId", StringType()),
        StructField("isNewCustomer", BooleanType()),
        StructField("ShippingAmount", DoubleType()),
        # InvoiceAddr / nameNoSurname are literal "<PII>" placeholder strings in the
        # sample data. In a real pipeline these should not be persisted downstream
        # un-redacted - flagged here, redaction itself is out of scope for this exercise.
        StructField("InvoiceAddr", StringType()),
        StructField("nameNoSurname", StringType()),
        StructField("carrier_id", StringType()),
        StructField("ExternOrderReference", StringType()),
        StructField("SalesChannel", StringType()),
        StructField("GuestFlag", StringType()),
        StructField("CustomerType", StringType()),
        StructField("CustomerSource", StringType()),
        StructField("Tags", StringType()),
        StructField("Cohort", StringType()),
        StructField("Segmentation", StringType()),
        StructField("OptIn", StringType()),
        StructField("InvoiceAmount", DoubleType()),
        StructField("PaymentMode", StringType()),
        StructField("OrderType", StringType()),
        StructField("LatestEvent", StringType()),
        StructField("LatestEventDate", StringType()),
        StructField("shortOrderNo", StringType()),
        StructField("totalPrice", DoubleType()),
        StructField("courierServiceLevel", StringType()),
        StructField("FirstPayment", StringType()),
        StructField("installmentPlanDisplay", ArrayType(StringType())),
        StructField("packslips", ArrayType(StringType())),
        # dynamically-numbered keys (primary_skn0, related_skn0, primary_skn1, ...)
        # with no fixed/bounded schema across rows - MapType avoids guessing a max index.
        StructField("productRecommendations", MapType(StringType(), StringType())),
    ])),
    StructField("currency", StringType()),
    StructField("currencyCode", StringType()),
    StructField("returnLabelsAdditional", ArrayType(StringType())),
    StructField("rmaStatus", StringType()),
    StructField("rmaStatusReason", StringType()),
    StructField("externalRMAId", StringType()),
    StructField("stateProvince", StringType()),
    StructField("returnPaymentInfo", StructType([
        StructField("amount", DoubleType()),
        StructField("currency", StringType()),
        StructField("pspReference", StringType()),
        StructField("merchantReference", StringType()),
    ])),
    StructField("returnCourierFee", StructType([
        StructField("value", DoubleType()),
        StructField("currency", StringType()),
    ])),
    StructField("pickupWindow", StringType()),
    StructField("pickupConfirmationCode", StringType()),
    StructField("totalPrice", DoubleType()),
    StructField("shortOrderNo", StringType()),
    StructField("pickupWindowFormatted", StringType()),
    # NOTE: from_json's default PERMISSIVE mode on a StructType schema (unlike
    # ArrayType) almost never returns a NULL struct for malformed-but-non-empty JSON -
    # it silently nulls out unparseable fields instead (verified empirically). The
    # only reliable way to detect a genuinely corrupt custom_fields record is to
    # include a _corrupt_record column and pass columnNameOfCorruptRecord.
    StructField("_corrupt_record", StringType()),
])
cf_fields = [f.name for f in custom_fields_schema.fields if f.name != "_corrupt_record"]


#### `clean_scalar_columns`

In [ ]:
# ==== clean_scalar_columns ====
def parse_timestamp_or_null(value, fmt):
    """Spark 4 runs in ANSI mode by default and raises on empty/malformed
    timestamps. The pipeline intentionally represents those source values as
    NULL instead."""
    cleaned = F.when(F.trim(value) == "", F.lit(None)).otherwise(value)
    return F.try_to_timestamp(cleaned, F.lit(fmt))


def clean_scalar_columns(df_raw):
    """Adds _row_uid and record_key, replaces literal 'null' sentinels with
    real NULLs, casts booleans/decimals/ints, and combines date+time column
    pairs into timestamps. Returns df_stage1 (uncached - caller decides
    caching). Pure transform: no prints/asserts beyond the contract checks
    below - diagnostics/profiling live in the calling execution cell.

    record_key = sha2(id || article_number) is the STABLE business key used
    downstream for the idempotent MERGE write - _row_uid
    (monotonically_increasing_id()) is only unique WITHIN a single run, is not
    stable across reruns, and must never be used as a persistent join/merge
    key. (id, article_number) is verified unique and non-null for this sample
    in the Task 1.1 execution cell before record_key is trusted downstream."""
    df_raw = df_raw.withColumn("_row_uid", F.monotonically_increasing_id())
    df_raw = df_raw.withColumn(
        "record_key",
        F.sha2(F.concat_ws("||", F.col("id"), F.col("article_number")), 256)
    )

    # ---- step 1: global "null"-string -> real NULL, over every string column ----
    df = df_raw.select(
        [F.when(F.trim(F.col(c)) == "null", None).otherwise(F.col(c)).alias(c) for c in df_raw.columns]
    )

    # ---- step 2: boolean casts ----
    for c in BOOL_COLS:
        df = df.withColumn(c, F.when(F.col(c).isNull(), None).otherwise(F.col(c) == F.lit("t")))

    # ---- step 3: date/time pair combination ----
    # NOTE: the paired *_time columns (order_time, pickup_scheduled_time, inbound_scan_time,
    # qualified_delivery_attempt_time, delivery_time, created_time) all carry seconds
    # (HH:mm:ss), verified live against the raw file - NOT HH:mm as originally assumed.
    for date_col, time_col, out_col in DATE_TIME_PAIRS:
        df = df.withColumn(
            out_col,
            parse_timestamp_or_null(
                F.concat_ws(" ", F.col(date_col), F.col(time_col)), "dd/MM/yyyy HH:mm:ss"
            )
        )

    # parcellab pairing - documented assumption; created_time also carries seconds
    df = df.withColumn(
        "parcellab_created_ts",
        parse_timestamp_or_null(
            F.concat_ws(" ", F.col("parcellab_system_created_date"), F.col("created_time")),
            "dd/MM/yyyy HH:mm:ss",
        ),
    )

    # updated / activity_monitor_last_update are HH:mm (no seconds). activity_monitor_last_update
    # additionally has 27/1000 corrupted values (same truncated-timestamp pattern as
    # refreshed_date, e.g. "15:08.8") - with timeParserPolicy=CORRECTED these fail to
    # match the pattern and become NULL rather than raising, which is the desired behavior
    # (silently losing 27 unparseable values is acceptable/expected here, same judgement
    # call as refreshed_date; not force-fixed).
    for c, out_col in SINGLE_TS_COLS.items():
        df = df.withColumn(out_col, parse_timestamp_or_null(F.col(c), "dd/MM/yyyy HH:mm"))

    for c, out_col in DATE_ONLY_COLS.items():
        df = df.withColumn(
            out_col,
            F.to_date(parse_timestamp_or_null(F.col(c), "dd/MM/yyyy")),
        )

    # ---- step 4: article_price decimal ----
    df = df.withColumn("article_price", F.col("article_price").cast(DecimalType(12, 2)))

    # ---- step 5: n_days_*/delta_* -> IntegerType (after null-cleanup, all are safe) ----
    for c in N_DAYS_COLS:
        df = df.withColumn(c, F.col(c).cast(IntegerType()))

    # ---- step 6: string-preserving columns must stay string (metadata-only check,
    # no Spark job triggered - fine to assert inside the function as a contract check) ----
    dtypes = dict(df.dtypes)
    for c in STRING_PRESERVE_COLS:
        assert dtypes[c] == "string", f"{c} expected string, got {dtypes[c]}"

    return df


#### `parse_articles`, `parse_custom_fields`

In [ ]:
# ==== parse_articles, parse_custom_fields ====
def repair_json_col(col):
    """Backslash-escaped commas inside string values break naive from_json -
    this regex only touches "\\," and doesn't corrupt the legitimate "\\n"/"\\""
    escapes also present in the data."""
    return F.regexp_replace(col, r"\\,", ",")


def parse_articles(df, articles_col="articles", article_number_col="article_number", row_key="_row_uid"):
    """Repairs escaped commas, parses the articles JSON array, matches the
    element where articleNo == article_number, and left-joins the flattened
    match back onto df by row_key - this preserves row count even for future
    files with 0-match or >1-match rows (both stay visible via
    articles_match_count rather than being silently dropped or fanned out)."""
    df_r = df.withColumn("articles_repaired", repair_json_col(F.col(articles_col)))
    df_a = df_r.withColumn("articles_arr", F.from_json(F.col("articles_repaired"), articles_schema))
    parse_failed = df_a.select(row_key, (F.col(articles_col).isNotNull() & F.col("articles_arr").isNull()).alias("articles_parse_failed"))
    exploded = df_a.select(row_key, article_number_col, F.explode_outer("articles_arr").alias("article_obj"))
    matched = exploded.filter(F.col("article_obj.articleNo") == F.col(article_number_col))
    match_counts = matched.groupBy(row_key).count().withColumnRenamed("count", "articles_match_count")
    matched_flat = matched.select(
        row_key, *[F.col(f"article_obj.{f}").alias(f"articles_json_{f}") for f in item_fields]
    ).dropDuplicates([row_key])
    out = (
        df.join(matched_flat, on=row_key, how="left")
          .join(match_counts, on=row_key, how="left")
          .join(parse_failed, on=row_key, how="left")
          .withColumn("articles_match_count", F.coalesce(F.col("articles_match_count"), F.lit(0)))
    )
    return out


def parse_custom_fields(df, custom_fields_col="custom_fields"):
    """Repairs escaped commas, parses custom_fields with corrupt-record
    detection (from_json's default PERMISSIVE mode doesn't null out a
    malformed StructType the way it does for ArrayType, so a _corrupt_record
    field + columnNameOfCorruptRecord is the only reliable detection method),
    and flattens top-level keys as custom_fields_json_*."""
    df_r = df.withColumn("custom_fields_repaired", repair_json_col(F.col(custom_fields_col)))
    df_p = df_r.withColumn(
        "custom_fields_json",
        F.from_json(
            F.col("custom_fields_repaired"), custom_fields_schema,
            {"columnNameOfCorruptRecord": "_corrupt_record"},
        ),
    )
    df_p = df_p.withColumn(
        "custom_fields_parse_failed",
        F.col("custom_fields_json._corrupt_record").isNotNull()
    )
    for f in cf_fields:
        df_p = df_p.withColumn(f"custom_fields_json_{f}", F.col(f"custom_fields_json.{f}"))
    return df_p


#### `flag_exceptions`

In [ ]:
# ==== flag_exceptions ====
VALIDATION_ERROR_CODES = {
    "articles_parse_failed": "ARTICLES_PARSE_FAILED",
    "articles_no_match": "ARTICLES_NO_MATCH",
    "articles_duplicate_match": "ARTICLES_DUPLICATE_MATCH",
    "custom_fields_parse_failed": "CUSTOM_FIELDS_PARSE_FAILED",
}


def flag_exceptions(df):
    """Builds validation_errors: array<string> from independent checks (not a
    first-match-wins chain), so a row with multiple simultaneous problems
    shows all of them instead of only the first one detected. has_exception
    is derived from the array's size rather than tracked separately.

    NOTE: a row whose articles JSON failed to parse also mechanically resolves
    articles_match_count to 0 (parse_articles's coalesce(..., 0) fallback for
    an unparseable array) - so ARTICLES_NO_MATCH is only raised when the match
    genuinely came up empty on a *successfully parsed* array
    (articles_parse_failed == False), mirroring the same guard
    attempt_no_match_repair already uses for candidate selection. Without this
    guard, every parse-failure row would also pick up a spurious, permanently
    stuck ARTICLES_NO_MATCH entry, since attempt_no_match_repair's repair path
    explicitly excludes parse-failed rows and could never clear it."""
    return (
        df.withColumn(
            "validation_errors",
            F.array_compact(F.array(
                F.when(F.col("articles_parse_failed"), F.lit(VALIDATION_ERROR_CODES["articles_parse_failed"])),
                F.when(
                    (F.col("articles_match_count") == 0) & (F.col("articles_parse_failed") == False),
                    F.lit(VALIDATION_ERROR_CODES["articles_no_match"])
                ),
                F.when(F.col("articles_match_count") > 1, F.lit(VALIDATION_ERROR_CODES["articles_duplicate_match"])),
                F.when(F.col("custom_fields_parse_failed"), F.lit(VALIDATION_ERROR_CODES["custom_fields_parse_failed"])),
            )),
        )
        .withColumn("has_exception", F.size("validation_errors") > 0)
    )


#### `attempt_articles_repair`, `attempt_no_match_repair`, `apply_repairs`

In [ ]:
# ==== attempt_articles_repair, attempt_no_match_repair, apply_repairs ====
def attempt_articles_repair(df, row_key="_row_uid"):
    """For rows whose articles JSON failed to parse even after the backslash-comma
    repair, try a cheap best-effort fixup (append a likely-missing closing
    bracket/brace) and re-attempt the parse + match."""
    candidates = (
        df.filter(F.col("articles_parse_failed") == True)
          .withColumn("articles_repaired", repair_json_col(F.col("articles")))
          .select(row_key, "articles_repaired", "article_number")
    )
    if candidates.limit(1).count() == 0:
        return df.withColumn("_articles_repair_success", F.lit(False)).select(
            row_key, "_articles_repair_success",
            *[F.lit(None).alias(f"repaired_articles_json_{f}") for f in item_fields])

    fixed = candidates.withColumn("articles_fix_attempt", F.concat(F.col("articles_repaired"), F.lit("}]")))
    fixed = fixed.withColumn("articles_fix_arr", F.from_json(F.col("articles_fix_attempt"), articles_schema))
    exploded_fix = fixed.select(row_key, "article_number", F.explode_outer("articles_fix_arr").alias("obj"))
    matched_fix = exploded_fix.filter(F.col("obj.articleNo") == F.col("article_number"))
    match_ct = matched_fix.groupBy(row_key).count()
    unique_match = match_ct.filter("count = 1").select(row_key)
    matched_fix_unique = matched_fix.join(unique_match, on=row_key, how="inner")
    repaired_flat = matched_fix_unique.select(
        row_key,
        F.lit(True).alias("_articles_repair_success"),
        *[F.col(f"obj.{f}").alias(f"repaired_articles_json_{f}") for f in item_fields],
    )
    return repaired_flat


def attempt_no_match_repair(df, row_key="_row_uid"):
    """For rows with articles_match_count == 0 (parsed fine, no exact articleNo match),
    retry the match using a trimmed/case-insensitive comparison as a fallback."""
    candidates = (
        df.filter((F.col("articles_match_count") == 0) & (F.col("articles_parse_failed") == False))
          .withColumn("articles_repaired", repair_json_col(F.col("articles")))
          .select(row_key, "articles_repaired", "article_number")
    )
    if candidates.limit(1).count() == 0:
        return df.limit(0).select(row_key).withColumn("_no_match_repair_success", F.lit(False)) \
                  .select(row_key, "_no_match_repair_success",
                          *[F.lit(None).alias(f"repaired_articles_json_{f}") for f in item_fields])

    arr = candidates.withColumn("articles_arr", F.from_json(F.col("articles_repaired"), articles_schema))
    exploded = arr.select(row_key, "article_number", F.explode_outer("articles_arr").alias("obj"))
    fuzzy_matched = exploded.filter(
        F.upper(F.trim(F.col("obj.articleNo"))) == F.upper(F.trim(F.col("article_number")))
    )
    match_ct = fuzzy_matched.groupBy(row_key).count()
    unique_match = match_ct.filter("count = 1").select(row_key)
    fuzzy_unique = fuzzy_matched.join(unique_match, on=row_key, how="inner")
    repaired_flat = fuzzy_unique.select(
        row_key,
        F.lit(True).alias("_no_match_repair_success"),
        *[F.col(f"obj.{f}").alias(f"repaired_articles_json_{f}") for f in item_fields],
    )
    return repaired_flat


def apply_repairs(df, row_key="_row_uid"):
    """Attempts both repair strategies and merges results: overwrites the
    affected articles_json_* columns where a repair succeeded, and removes
    only the specific validation_errors code(s) each successful repair
    addressed - a row that had two independent problems and only got one
    fixed still shows the other, rather than being wrongly marked fully
    clean. ARTICLES_DUPLICATE_MATCH and CUSTOM_FIELDS_PARSE_FAILED are never
    removed here since no repair strategy exists for either. record_status
    (valid/repaired/invalid) is derived from has_exception's pre- and
    post-repair state."""
    parse_fix = attempt_articles_repair(df, row_key)
    no_match_fix = attempt_no_match_repair(df, row_key)

    df2 = df.join(
        parse_fix.select(row_key, "_articles_repair_success",
                          *[F.col(f"repaired_articles_json_{f}").alias(f"pf_{f}") for f in item_fields]),
        on=row_key, how="left",
    ).join(
        no_match_fix.select(row_key, "_no_match_repair_success",
                             *[F.col(f"repaired_articles_json_{f}").alias(f"nm_{f}") for f in item_fields]),
        on=row_key, how="left",
    )
    df2 = df2.withColumn("_articles_repair_success", F.coalesce(F.col("_articles_repair_success"), F.lit(False)))
    df2 = df2.withColumn("_no_match_repair_success", F.coalesce(F.col("_no_match_repair_success"), F.lit(False)))
    df2 = df2.withColumn("repair_attempted", F.col("has_exception"))

    # overwrite the affected articles_json_* columns where a repair succeeded
    for f in item_fields:
        df2 = df2.withColumn(
            f"articles_json_{f}",
            F.when(F.col("_articles_repair_success"), F.col(f"pf_{f}"))
             .when(F.col("_no_match_repair_success"), F.col(f"nm_{f}"))
             .otherwise(F.col(f"articles_json_{f}")),
        )

    # remove only the specific error code(s) each successful repair addressed -
    # F.filter (not array_except) preserves element order deterministically
    df2 = df2.withColumn(
        "validation_errors",
        F.when(
            F.col("_articles_repair_success"),
            F.filter(F.col("validation_errors"), lambda x: x != F.lit(VALIDATION_ERROR_CODES["articles_parse_failed"]))
        ).otherwise(F.col("validation_errors")),
    )
    df2 = df2.withColumn(
        "validation_errors",
        F.when(
            F.col("_no_match_repair_success"),
            F.filter(F.col("validation_errors"), lambda x: x != F.lit(VALIDATION_ERROR_CODES["articles_no_match"]))
        ).otherwise(F.col("validation_errors")),
    )
    df2 = df2.withColumn("has_exception", F.size("validation_errors") > 0)
    df2 = df2.withColumn(
        "record_status",
        F.when(F.col("has_exception"), F.lit("invalid"))
         .when(F.col("repair_attempted"), F.lit("repaired"))
         .otherwise(F.lit("valid"))
    )

    drop_cols = [f"pf_{f}" for f in item_fields] + [f"nm_{f}" for f in item_fields] \
        + ["_articles_repair_success", "_no_match_repair_success"]
    return df2.drop(*drop_cols)


#### `route_records`

In [ ]:
# ==== route_records ====
def route_records(df_stage5):
    """Splits df_stage5 into (df_success, df_unresolved) - the Good/Bad split.
    Bad (unresolved) rows keep a diagnostic column subset including
    validation_errors, record_status, and record_key (needed on this path too,
    since it's the idempotent MERGE key for the Bad table). Good is everything
    without a remaining exception - has_exception already reflects the
    POST-repair state (recomputed inside apply_repairs), so a plain
    has_exception check is sufficient; the old has_exception & ~repaired
    compound condition from the first-match-wins design is no longer needed."""
    diagnostic_cols = ["_row_uid", "record_key", "id", "article_number", "order_no",
                        "validation_errors", "record_status", "articles", "custom_fields",
                        "order_ts", "updated_ts"]
    df_unresolved = df_stage5.filter(F.col("has_exception")).select(*diagnostic_cols)
    df_success = df_stage5.filter(~F.col("has_exception"))
    return df_success, df_unresolved


#### `build_final_dataset`, `optimize_datatypes`, `validate_final_schema`

In [ ]:
# ==== build_final_dataset, optimize_datatypes, validate_final_schema ====
def build_final_dataset(df_success, df_stage1):
    """Builds final_col_order (every original df_stage1 column, which now
    includes record_key, + articles_json_* + custom_fields_json_* +
    validation columns) and selects it from df_success."""
    articles_json_cols = [c for c in df_success.columns if c.startswith("articles_json_")]
    custom_fields_json_cols = [c for c in df_success.columns if c.startswith("custom_fields_json_")]
    validation_cols = [
        "articles_match_count", "articles_parse_failed", "custom_fields_parse_failed",
        "validation_errors", "has_exception", "repair_attempted", "record_status",
    ]
    # base_cols = every original scalar column from df_stage1 (the 86 raw source
    # columns, retained unless justified otherwise) PLUS the additive combined
    # date/time columns and record_key added in clean_scalar_columns - both are
    # kept side by side (not a replacement) per the task's "retain all original
    # source columns" instruction.
    base_cols = df_stage1.columns
    final_col_order = list(dict.fromkeys(base_cols + articles_json_cols + custom_fields_json_cols + validation_cols))
    return df_success.select(*final_col_order)


def optimize_datatypes(df_final):
    """Casts articles_json_price/articlePrice/priceDetails to Decimal(12,2).
    Returns df_final_typed."""
    return (
        df_final
        .withColumn("articles_json_price", F.col("articles_json_price").cast(DecimalType(12, 2)))
        .withColumn("articles_json_articlePrice", F.col("articles_json_articlePrice").cast(DecimalType(12, 2)))
        .withColumn(
            "articles_json_priceDetails",
            F.struct(
                F.col("articles_json_priceDetails.pricePaid").cast(DecimalType(12, 2)).alias("pricePaid"),
                F.col("articles_json_priceDetails.pricePaidNoTax").cast(DecimalType(12, 2)).alias("pricePaidNoTax"),
            ),
        )
    )


def validate_final_schema(df_final_typed):
    """Runs the schema assertions (string-preserve, bool, int, decimal cols +
    is_/has_ heuristic). Raises AssertionError on failure; returns the
    "suspect" list (empty on success) for the caller to print."""
    dtypes = dict(df_final_typed.dtypes)

    for c in STRING_PRESERVE_COLS:
        assert dtypes[c] == "string", f"{c} expected string, got {dtypes[c]}"

    for c in BOOL_COLS:
        assert dtypes[c] == "boolean", f"{c} expected boolean, got {dtypes[c]}"

    for c in N_DAYS_COLS:
        assert dtypes[c] == "int", f"{c} expected int, got {dtypes[c]}"

    assert dtypes["article_price"].startswith("decimal"), dtypes["article_price"]
    assert dtypes["articles_json_price"].startswith("decimal"), dtypes["articles_json_price"]

    # heuristic: flag any is_/has_-prefixed column still typed as plain string
    # (raw source *_date/*_time columns are expected to stay string - they're
    # kept alongside the combined _ts/_dt columns for traceability, not replaced)
    suspect = [
        c for c, t in dtypes.items()
        if t == "string" and (c.startswith("is_") or c.startswith("has_"))
    ]
    assert suspect == []
    return suspect


#### `write_outputs`, `build_run_audit_record`, `build_summary_metrics`

In [ ]:
# ==== write_outputs, build_run_audit_record, write_run_audit, build_summary_metrics ====
def _sql_str(s):
    """Escapes a value for safe interpolation into a SQL string literal
    (doubles embedded single quotes). Applied to internally-generated values
    (UUIDs, configured table names) before building VALUES literals - not
    untrusted user input, but cheap and correct to do properly regardless."""
    return str(s).replace("'", "''")


def write_outputs(df_final_typed, df_unresolved, good_table, bad_table, pipeline_run_id):
    """Idempotent upsert into the Good/Bad Delta tables, keyed by record_key -
    NOT overwrite, so re-running the pipeline against the same source data
    does not create duplicate rows or wipe out history from prior daily runs.
    First run per table (table doesn't exist yet): plain saveAsTable.
    Subsequent runs: MERGE. first_seen_run_id/first_seen_at are populated only
    via whenNotMatchedInsertAll() on initial insert and are never touched by
    the update clause, so they track true first-seen provenance rather than
    "last touched by" - which a naive whenMatchedUpdateAll() would collapse
    them into. last_seen_run_id/last_updated_at are refreshed on every match.

    Table size is no longer 1000-rows-and-done: it accumulates across runs, so
    partitioning (e.g. by ingestion date) becomes worth revisiting once
    multiple days of history build up - not needed yet at this volume."""
    now = F.current_timestamp()
    stamp_cols = ("first_seen_run_id", "first_seen_at", "last_seen_run_id", "last_updated_at")

    def stamp(df):
        return (
            df
            .withColumn("first_seen_run_id", F.lit(pipeline_run_id))
            .withColumn("first_seen_at", now)
            .withColumn("last_seen_run_id", F.lit(pipeline_run_id))
            .withColumn("last_updated_at", now)
        )

    def upsert(source_df, table_name):
        if not spark.catalog.tableExists(table_name):
            source_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
            return
        target = DeltaTable.forName(spark, table_name)
        data_cols = [c for c in source_df.columns if c not in stamp_cols]
        update_set = {c: f"s.{c}" for c in data_cols}
        update_set["last_seen_run_id"] = "s.last_seen_run_id"
        update_set["last_updated_at"] = "s.last_updated_at"
        (
            target.alias("t")
            .merge(source_df.alias("s"), "t.record_key = s.record_key")
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsertAll()
            .execute()
        )

    upsert(stamp(df_final_typed), good_table)
    upsert(stamp(df_unresolved), bad_table)

    n_good_rb = spark.table(good_table).count()
    n_bad_rb = spark.table(bad_table).count()
    return n_good_rb, n_bad_rb


def build_run_audit_record(pipeline_run_id, source_table, source_table_version,
                            started_at, completed_at, input_rows, good_rows, bad_rows, status):
    """Builds a single-row DataFrame for pipeline_run_audit. Built via SQL
    VALUES for the same portability reason as build_summary_metrics (avoids
    spark.createDataFrame(list, ...), which round-trips through a Python
    worker for RDD-based parallelize/serialization and is a known crash
    source on some local pyspark/Python/OS combinations)."""
    version_literal = "NULL" if source_table_version is None else f"'{_sql_str(source_table_version)}'"
    return spark.sql(f"""
        SELECT * FROM VALUES (
            '{_sql_str(pipeline_run_id)}', '{_sql_str(source_table)}', {version_literal},
            TIMESTAMP'{started_at.isoformat(sep=" ")}',
            TIMESTAMP'{completed_at.isoformat(sep=" ")}',
            {input_rows}L, {good_rows}L, {bad_rows}L, '{_sql_str(status)}'
        ) AS t(pipeline_run_id, source_table, source_table_version, started_at,
               completed_at, input_rows, good_rows, bad_rows, status)
    """)


def write_run_audit(audit_df, audit_table):
    """Appends one row to the (append-only) run-audit table, creating it on
    first use.

    KNOWN LIMITATION: this is called once, at the very end of a successful
    run - a run that throws partway through leaves no audit row at all for
    that attempt, rather than a row marked FAILED/RUNNING. A more complete
    design would insert a RUNNING row up front and update it on completion;
    documented here as a deliberate scope cut for this exercise rather than
    built, consistent with how the source-file-identity gap above is handled
    (state the limitation honestly rather than over-build or fake it)."""
    if not spark.catalog.tableExists(audit_table):
        audit_df.write.format("delta").mode("overwrite").saveAsTable(audit_table)
    else:
        audit_df.write.format("delta").mode("append").saveAsTable(audit_table)


def build_summary_metrics(total_input_rows, requiring_exception_handling, successfully_repaired,
                           still_unresolved, successfully_processed):
    """Builds the SQL-VALUES-based summary_df and runs the reconciliation
    asserts. Returns summary_df."""
    summary_df = spark.sql(f"""
        SELECT * FROM VALUES
            ('Total input rows', {total_input_rows}L),
            ('Successfully processed rows', {successfully_processed}L),
            ('Rows requiring exception handling', {requiring_exception_handling}L),
            ('Rows successfully repaired', {successfully_repaired}L),
            ('Rows still unresolved', {still_unresolved}L)
        AS t(metric, value)
    """)

    # total_input_rows is checked as non-empty, not pinned to any specific count -
    # future daily files may legitimately have a different row count than this
    # sample. See Task 1.1 for the sample-specific 1000-row observation (a print,
    # not an assert).
    check_a = total_input_rows > 0
    check_b = successfully_processed + still_unresolved == total_input_rows
    check_c = requiring_exception_handling == successfully_repaired + still_unresolved
    assert check_a and check_b and check_c, (check_a, check_b, check_c)

    return summary_df


***
## Part 1: Load and Inspect the Source Data. Process Scalar columns




The dataset `daily_returns_trackings_2026-07-01.csv` contains raw data with 86 columns.



### Task 1.1: Ingest the raw file & profile raw data

Read the source CSV file into Spark and inspect the schema. Perform an initial review of the data and understand if there are any transformation required for scalar columns to make them readable and ready for analytical database. 

_**Hint** : Focus on columns having correct datatype and formats that can be later converted into relational tables. (Date and time columns)_



In [ ]:
# ==== Task 1.1 execution: read input, profile raw data, clean_scalar_columns ====
df_raw = spark.table(INPUT_TABLE)
n_raw = df_raw.count()
print("raw row count:", n_raw)
print("raw col count:", len(df_raw.columns))
# The real precondition is a non-empty input, not any specific row count - a future
# daily file may legitimately have a different number of rows than this sample.
assert n_raw > 0, "Input table is empty"
if n_raw != 1000:
    print(f"NOTE: this sample file was profiled against exactly 1000 rows; got {n_raw}. "
          f"The row-count checks throughout this notebook are relative to n_raw (this "
          f"run's actual input size), not hard-coded to 1000, so they remain valid here.")

# ---- profile: literal "null" string prevalence ----
null_str_counts = {}
for c in df_raw.columns:
    cnt = df_raw.filter(F.trim(F.col(c)) == "null").count()
    if cnt > 0:
        null_str_counts[c] = cnt
print("\ncolumns with literal 'null' string (col: count):")
for k, v in sorted(null_str_counts.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

# check each n_days col for non-numeric, non-"null" garbage before the blanket cast
# inside clean_scalar_columns
print("\nn_days_* / delta_* distinct-shape audit:")
for c in N_DAYS_COLS:
    bad = df_raw.filter(
        F.col(c).isNotNull() & (F.trim(F.col(c)) != "null") & (~F.col(c).rlike(r"^-?\d+$"))
    )
    bad_cnt = bad.count()
    print(f"  {c}: non-numeric-non-null count = {bad_cnt}")
    if bad_cnt > 0:
        bad.select(c).show(5, truncate=False)

df_stage1 = clean_scalar_columns(df_raw).cache()

# record_key precondition: (id, article_number) must be unique and non-null for
# record_key (sha2 of the two concatenated) to be safe as a MERGE key downstream.
# concat_ws silently SKIPS null arguments (unlike concat), so a null id would not
# null out record_key - it would produce a key that could collide with an
# unrelated row. Checked explicitly rather than assumed from today's clean sample.
key_nulls = df_stage1.filter(F.col("id").isNull() | F.col("article_number").isNull()).count()
key_dupes = df_stage1.groupBy("id", "article_number").count().filter("count > 1").limit(1).count()
print(f"\nrecord_key precondition: {key_nulls} null (id, article_number) rows, "
      f"{'no' if key_dupes == 0 else 'FOUND'} duplicate (id, article_number) pairs")
assert key_nulls == 0, "id/article_number must be non-null to use record_key as a merge key"
assert key_dupes == 0, "id/article_number must be unique to use record_key as a merge key"

# reconciliation check: sentinel "null" strings found in the raw column vs. real
# nulls present in the same column after clean_scalar_columns - proves nothing was
# silently lost or over-nulled by the later casts
total_null_str = sum(null_str_counts.values())
after_nulls = sum(
    df_stage1.filter(F.col(c).isNull()).count() - df_raw.filter(F.col(c).isNull()).count()
    for c in null_str_counts.keys()
)
print(f"\nreconciliation: total 'null'-string sentinels found = {total_null_str}, "
      f"new real-nulls introduced = {after_nulls}")
assert total_null_str == after_nulls, "mismatch between 'null' strings found and nulls introduced"

print("\nstring-preserving columns confirmed still StringType:", STRING_PRESERVE_COLS)

# sample values to confirm no leading-zero loss
df_stage1.select("customer_no", "id", "order_no").show(5, truncate=False)

# refreshed_date: leave raw, just show sample unparseable values
print("\nrefreshed_date sample values (left unparsed, corrupted/truncated):")
df_stage1.select("refreshed_date").filter(F.col("refreshed_date").isNotNull()).show(5, truncate=False)

activity_corrupt_cnt = df_stage1.filter(
    F.col("activity_monitor_last_update").isNotNull()
    & ~F.col("activity_monitor_last_update").rlike(r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$")
).count()
print(f"\nactivity_monitor_last_update: {activity_corrupt_cnt} corrupted/truncated values "
      f"(same pattern as refreshed_date) - these become NULL in activity_monitor_last_update_ts")

n_stage1 = df_stage1.count()
print("\ndf_stage1 row count:", n_stage1)
assert n_stage1 == n_raw

print("\ndf_stage1 schema:")
df_stage1.printSchema()


**Narrative Question (1.1): Briefly summarise the key transformation challenges you identified in the source file.**

1. **Null handling:** The source uses the literal string `"null"` instead of real null values in many optional fields. I converted these values to proper nulls before applying any type conversions, so they are handled consistently in later transformations.

2. **Date and timestamp parsing:** Most date fields use `dd/MM/yyyy` and several timestamps are split across separate date and time columns. These were combined and converted into timestamps using the appropriate format. Some values in `activity_monitor_last_update` and `refreshed_date` are malformed, so they are left as null/raw values rather than being guessed or repaired.

3. **Boolean conversion:** Some columns store yes/no values as `'t'` and `'f'`. I converted these into proper `True` and `False` Boolean values.

4. **Numeric precision:** `article_price` contains floating-point precision artefacts, so it was converted to `DecimalType` rather than `DoubleType`.

5. **Identifier fields:** Columns such as `id`, `customer_no`, `order_no`, `tracking_number`, and `delivery_no` were retained as strings to avoid losing leading zeros or changing identifier values.

***
## Part 2: Transform the _articles_ and _custom_fields_ Column



### Task 2.1: Parse and match the correct article object
For each row:

1. Parse the _articles_ column
2. Identify the single JSON object where _articleNo_ matches the rowÃ¢â‚¬â„¢s _article_number_
3. flatten only that matched JSON object into new columns
4. Prefix the flattened columns with: articles_json_
5. Retain the original raw articles column in the output for traceability and auditability.
6. Add useful validation checks that help identify whether parsing and matching worked correctly.

**Note :** Your transformation must keep the record count unchanged throughout this process.




In [ ]:
# ==== Task 2.1 execution: parse_articles ====
df_stage2 = parse_articles(df_stage1).cache()

n_stage2 = df_stage2.count()
print("df_stage2 row count (must stay == n_raw):", n_stage2, " n_raw:", n_raw)
assert n_stage2 == n_raw

match_dist_final = dict(df_stage2.groupBy("articles_match_count").count().collect())
print("articles_match_count distribution (validation check - expect {1: n_raw} on a clean file):", match_dist_final)

df_stage2.select("_row_uid", "id", "article_number", "articles_json_articleNo",
                  "articles_json_price", "articles_match_count").show(5, truncate=False)

# df_stage1's row data is never touched again after this point - only its .columns
# (schema metadata, unaffected by unpersist) is read later in build_final_dataset.
df_stage1.unpersist()


**Narrative Question (2.1): Explain how you ensured that only the correct article was flattened while keeping the row count unchanged.**

1. **Unique row key:** The `id` column is not unique in this dataset; the same shipment can contain multiple returned articles. I created `_row_uid` immediately after ingestion and used it as the row-level key for joins and validation.

2. **Correct article match:** I parsed the `articles` JSON array, exploded it temporarily, and matched each object where `articleNo` equals the row's `article_number`.

3. **Validation before flattening:** I calculated the number of matches for every `_row_uid` before joining the result back. In this dataset, every row had exactly one matching article.

4. **Preserving the row count:** The matched article data was left-joined back to the original, unexpanded dataframe using `_row_uid`. This keeps all original rows, even if a future file contains no matching article.

5. **Flattening approach:** The matching article object was flattened into `articles_json_*` columns. Nested objects such as `customFields`, `priceDetails`, and `tracking` were retained as structured columns rather than being expanded further.

### Task 2.2: Transform the custom_fields Column

For each row:

1. Parse the custom_fields column and flatten the nested fields into new columns.
2. Prefix the flattened columns with: custom_fields_json_
3. Retain the original raw custom_fields column in the output for traceability and auditability.
4. Add useful validation fields to help identify whether parsing worked correctly.

**Note** : For nested json objects inside custom_fields, avoid opening objects with lists




In [ ]:
# ==== Task 2.2 execution: parse_custom_fields ====
df_stage3 = parse_custom_fields(df_stage2).cache()

parse_failed_count = df_stage3.filter("custom_fields_parse_failed").count()
print("custom_fields parse failures (validation check, expect 0 - schema verified against this file):", parse_failed_count)

n_stage3 = df_stage3.count()
print("df_stage3 row count (must stay == n_raw):", n_stage3, " n_raw:", n_raw)
assert n_stage3 == n_raw
print("custom_fields_parse_failed count:", df_stage3.filter("custom_fields_parse_failed").count())

# df_stage2's row data is never touched again after this point.
df_stage2.unpersist()


**Narrative Question (2.2): Briefly describe your approach to flattening `custom_fields` and handling any malformed rows.**

1. **Parsing approach:** `custom_fields` contains one JSON object per row. I cleaned the escaped-comma issue, then parsed the JSON using an explicit schema based on the fields found in the sample data.

2. **Validation:** I validated the schema against all 1,000 rows. No parsing failures were found in this sample, and the parsed fields were checked against the raw JSON values.

3. **Malformed JSON handling:** I included a `_corrupt_record` field in the parsing schema. This allows malformed JSON rows to be detected and routed to the exceptions process instead of silently producing null fields.

4. **Flattening approach:** Top-level fields were flattened into `custom_fields_json_*` columns. Nested objects were retained as structured columns where further flattening would create unnecessary complexity.

5. **Arrays and dynamic fields:** Array values such as `packslips`, `returnLabelsAdditional`, and `shopifyOrderTags` were kept as arrays. Dynamically named fields in `productRecommendations` were stored as a map rather than assuming a fixed schema.

6. **PII consideration:** Some nested fields contain `"<PII>"` placeholders. In a production pipeline, these fields should be reviewed and protected before being made available downstream.

***
## Part 3: Exception Handling




### Task 3.1: Identify problematic rows

Create logic to identify rows where:

1. Parsing failed
2. The articles match could not be found
3. Duplicate matches were found or other transformation issues occurred



In [ ]:
# ==== Task 3.1 execution: flag_exceptions + synthetic bad-row test harness ====
df_stage4 = flag_exceptions(df_stage3).cache()

n_exceptions_real = df_stage4.filter("has_exception").count()
print("real-data exception count (expected 0 - every row matched exactly once, both schemas parse clean):", n_exceptions_real)

# ---------------- synthetic bad-row test harness ----------------
# This sample file has zero organic exception rows, so the only way to prove this
# logic actually works is to exercise it against fabricated bad rows, reusing the
# exact same parse_articles/parse_custom_fields/flag_exceptions functions above
# rather than reimplementing anything:
#   SYN-A: articles JSON that stays malformed even after the backslash-comma repair
#          (missing closing bracket) -> parse fails
#   SYN-B: valid articles array, but no element's articleNo matches article_number at all
#   SYN-C: valid articles (1 clean match) but custom_fields JSON is malformed
#   SYN-D: articleNo differs from article_number only by whitespace/case -> repairable
#   SYN-E: valid articles array, genuinely no match, no near-match either -> unrepairable
#   SYN-F: BOTH articles JSON malformed (fixable) AND custom_fields JSON malformed
#          (unfixable) at once - proves a partial repair leaves the untouched error
#          visible in validation_errors rather than wrongly marking the row clean.
#          None of SYN-A..E exercise this since they're all single-error.
# Built via SQL VALUES (not spark.createDataFrame(list, ...)) - stays entirely on the
# JVM side and is the more portable construction method for a Databricks notebook.
df_exception_demo_raw = spark.sql(r"""
    SELECT * FROM VALUES
        ('SYN-A', 1001L, '[{"articleNo":"X1","price":10.0', 'X1', '{"isAdditionalLabel":false}'),
        ('SYN-B', 1002L, '[{"articleNo":"X1","price":10.0}]', 'DOES-NOT-EXIST', '{"isAdditionalLabel":false}'),
        ('SYN-C', 1003L, '[{"articleNo":"X1","price":10.0}]', 'X1', '{"isAdditionalLabel":false'),
        ('SYN-D', 1004L, '[{"articleNo":" x1 ","price":10.0}]', 'X1', '{"isAdditionalLabel":false}'),
        ('SYN-E', 1005L, '[{"articleNo":"Q9","price":10.0}]', 'X1', '{"isAdditionalLabel":false}'),
        ('SYN-F', 1006L, '[{"articleNo":"F1","price":10.0', 'F1', '{"isAdditionalLabel":false')
    AS t(id, _row_uid, articles, article_number, custom_fields)
""")

df_demo = parse_articles(df_exception_demo_raw)
df_demo = parse_custom_fields(df_demo)
df_demo = flag_exceptions(df_demo)

print("\nsynthetic exception-demo results:")
df_demo.select("id", "articles_parse_failed", "articles_match_count", "custom_fields_parse_failed", "validation_errors").show(truncate=False)

demo_errors = {r["id"]: r["validation_errors"] for r in df_demo.select("id", "validation_errors").collect()}
print("demo validation_errors:", demo_errors)
assert demo_errors["SYN-A"] == ["ARTICLES_PARSE_FAILED"], demo_errors
assert demo_errors["SYN-B"] == ["ARTICLES_NO_MATCH"], demo_errors
assert demo_errors["SYN-C"] == ["CUSTOM_FIELDS_PARSE_FAILED"], demo_errors
assert demo_errors["SYN-F"] == ["ARTICLES_PARSE_FAILED", "CUSTOM_FIELDS_PARSE_FAILED"], demo_errors

# df_stage3's row data is never touched again after this point - df_stage4 (cached
# above) now carries everything Task 3.2 needs.
df_stage3.unpersist()


### Task 3.2: Correct exception rows where possible
Where possible, attempt to correct and reprocess exception rows rather than only flagging them.



In [ ]:
# ==== Task 3.2 execution: apply_repairs ====
df_stage5 = apply_repairs(df_stage4)
df_stage5 = df_stage5.cache()
n_stage5 = df_stage5.count()
print("df_stage5 row count (must stay == n_raw):", n_stage5, " n_raw:", n_raw)
assert n_stage5 == n_raw
print("real-data repaired count (expect 0, nothing to repair):", df_stage5.filter("record_status = 'repaired'").count())
print("real-data still-has_exception count (expect 0):", df_stage5.filter("has_exception").count())

# ---- demonstrate against the synthetic exception rows ----
df_demo_repaired = apply_repairs(df_demo, row_key="_row_uid")
print("\nsynthetic repair-demo results:")
df_demo_repaired.select("id", "validation_errors", "has_exception", "repair_attempted", "record_status").orderBy("id").show(truncate=False)

demo_repair_state = {
    r["id"]: (r["validation_errors"], r["record_status"])
    for r in df_demo_repaired.select("id", "validation_errors", "record_status").collect()
}
print("demo repair state:", demo_repair_state)

# SYN-A: articles JSON was truncated but fixable by appending "}]" -> fully repaired
assert demo_repair_state["SYN-A"] == ([], "repaired"), demo_repair_state
# SYN-D: case/whitespace mismatch -> fixable via fuzzy match -> fully repaired
assert demo_repair_state["SYN-D"] == ([], "repaired"), demo_repair_state
# SYN-B, SYN-E: genuinely no matching article at all -> cannot be repaired
assert demo_repair_state["SYN-B"] == (["ARTICLES_NO_MATCH"], "invalid"), demo_repair_state
assert demo_repair_state["SYN-E"] == (["ARTICLES_NO_MATCH"], "invalid"), demo_repair_state
# SYN-C: custom_fields totally malformed, no repair strategy exists for it -> stays invalid
assert demo_repair_state["SYN-C"] == (["CUSTOM_FIELDS_PARSE_FAILED"], "invalid"), demo_repair_state
# SYN-F: the articles half gets fixed, but the custom_fields half doesn't - proves a
# partial repair leaves the untouched error visible rather than marking the row
# wrongly clean. This is the core assertion this whole feature is about.
assert demo_repair_state["SYN-F"] == (["CUSTOM_FIELDS_PARSE_FAILED"], "invalid"), demo_repair_state

print("\nAt least one synthetic row fully repaired (SYN-A, SYN-D), at least one stays genuinely "
      "unresolved (SYN-B, SYN-C, SYN-E), and one partially-repaired row (SYN-F) correctly keeps "
      "its remaining error visible instead of being marked clean - repair logic proven end to end.")

# df_stage4's row data is never touched again after this point.
df_stage4.unpersist()


### Task 3.3: Route unresolved records
Any rows that still cannot be processed correctly should be written to a separate exceptions output with useful diagnostic fields.



In [ ]:
# ==== Task 3.3 execution: route_records ====
df_success, df_unresolved = route_records(df_stage5)

n_unresolved = df_unresolved.count()
n_success = df_success.count()
print("df_success count:", n_success, " df_unresolved count:", n_unresolved, " n_raw:", n_raw)
assert n_success + n_unresolved == n_raw
print("real-data unresolved (expect 0):", n_unresolved)

# demonstrate routing on the synthetic set end-to-end (route_records itself isn't
# reused here since its diagnostic_cols include order_no/order_ts/updated_ts, which
# don't exist on the synthetic demo rows - same distinction the original notebook made)
demo_unresolved = df_demo_repaired.filter(F.col("has_exception"))
demo_success = df_demo_repaired.filter(~F.col("has_exception"))
print("\nsynthetic routing demo:")
print("  success:", [r["id"] for r in demo_success.select("id").orderBy("id").collect()])
print("  unresolved:", [r["id"] for r in demo_unresolved.select("id").orderBy("id").collect()])
assert demo_success.count() + demo_unresolved.count() == df_demo_repaired.count()
assert set(r["id"] for r in demo_unresolved.select("id").collect()) == {"SYN-B", "SYN-C", "SYN-E", "SYN-F"}
assert set(r["id"] for r in demo_success.select("id").collect()) == {"SYN-A", "SYN-D"}


__Narrative Question 3:__ What kinds of exception cases did you encounter, and how did you decide whether to repair or isolate them?

This sample file has **zero organic exception rows** - every row's `articles` and `custom_fields` JSON parses cleanly, and every row matches exactly one article. That made it impossible to validate the exception-handling logic against real failures, so a small synthetic harness (5 fabricated rows, reusing the exact same repair/parse functions as the main pipeline - not a reimplementation) was built to exercise every exception path the task describes, since the notebook explicitly warns future daily files may exhibit edge cases this sample doesn't:

- **Articles JSON parse failure** (truncated/malformed JSON even after the backslash-comma repair) - a best-effort secondary repair is attempted (appending a plausibly-missing closing bracket/brace and re-parsing); if that recovers a valid, uniquely-matching article, the row is marked repaired. If not, it's routed to the exceptions output with the raw JSON retained for diagnosis.
- **No matching article** (`articles_match_count == 0`) - a fallback trimmed/case-insensitive comparison between `articleNo` and `article_number` is attempted first (catches whitespace/casing mismatches); if that still finds no unique match, the row is genuinely unresolvable and routed to exceptions.
- **Duplicate matches** (`articles_match_count > 1`) - flagged but not automatically resolved, since picking one of several equally-valid matches would be a data integrity risk rather than a real repair; these are isolated for manual review.
- **custom_fields parse failure** - detected via the `_corrupt_record` mechanism (see Narrative 2.2); no automatic repair is attempted here since a malformed nested JSON object doesn't have an obvious cheap fixup the way a missing trailing bracket does, so these are isolated directly.

The general principle: attempt a **cheap, low-risk, deterministic** repair only where one clearly exists (a likely-truncated bracket, a whitespace/case mismatch); anything requiring a judgement call about *which* value is correct (duplicate matches, genuinely malformed nested objects) is isolated to the exceptions output with full diagnostic context (raw `articles`/`custom_fields`, `exception_type`, key identifiers) rather than guessed at automatically.



***
## Part 4: Build the Final Dataset



### Task 4.1: Produce the final transformed dataset

Create a final dataset that:

1. Retains all original source columns unless you justify otherwise
2. Retains raw complex columns
3. Includes flattened articles_json_* columns
4. Includes flattened custom_fields_json_* columns
5. Includes useful validation / debug columns




In [ ]:
# ==== Task 4.1 execution: build_final_dataset ====
df_final = build_final_dataset(df_success, df_stage1)

print("final column count:", len(df_final.columns))

n_final = df_final.count()
# build_final_dataset is a pure column SELECT on df_success (no row filtering), so
# it must have exactly as many rows as df_success - NOT n_raw, since df_success
# already excludes unresolved rows.
print("df_final row count:", n_final, " n_success:", n_success)
assert n_final == n_success

# spot-check articles_json_* / custom_fields_json_* against the raw JSON for one row
sample = df_final.select("_row_uid", "id", "article_number", "articles_json_articleNo",
                          "articles_json_price", "custom_fields_json_isAdditionalLabel",
                          "custom_fields_json_originalCourier").limit(3)
sample.show(truncate=False)


### Task 4.2: Optimise datatypes
Store the final dataset with sensible data types to support downstream performance and usability.



In [ ]:
# ==== Task 4.2 execution: optimize_datatypes + validate_final_schema ====
df_final_typed = optimize_datatypes(df_final)

n_typed = df_final_typed.count()
# optimize_datatypes only casts types, doesn't filter rows - must match df_final.
print("df_final_typed row count:", n_typed, " n_final:", n_final)
assert n_typed == n_final

suspect = validate_final_schema(df_final_typed)
print("\nboolean-prefixed columns still string (should be empty):", suspect)

print("\nfinal schema (df_final_typed):")
df_final_typed.printSchema()


### Task 4.3: Save Final Dataset in the proper format
Store the final dataset with proper format and file name.

**Note:** Send the final dataset and exception dataset (if applicable) along with your solution databricks notebook.



In [ ]:
# ==== Task 4.3 execution: write_outputs (idempotent MERGE) + run audit ====
GOOD_TABLE = f"{OUTPUT_SCHEMA}.returns_tracking_good"
BAD_TABLE = f"{OUTPUT_SCHEMA}.returns_tracking_bad"
AUDIT_TABLE = f"{OUTPUT_SCHEMA}.pipeline_run_audit"

n_good_rb, n_bad_rb = write_outputs(df_final_typed, df_unresolved, GOOD_TABLE, BAD_TABLE, PIPELINE_RUN_ID)

# >= rather than == : this run's data is upserted by record_key, not appended
# blindly, so a table that already has history from prior runs will have MORE rows
# than just this run's frame once genuinely new keys accumulate across days - it
# will never have fewer. On the very first run (table didn't exist before), this
# is exactly equal.
print("read-back good count:", n_good_rb, " this run's good rows:", df_final_typed.count())
print("read-back bad count:", n_bad_rb, " this run's bad rows:", df_unresolved.count())
assert n_good_rb >= df_final_typed.count()
assert n_bad_rb >= df_unresolved.count()

pipeline_completed_at = datetime.utcnow()
audit_df = build_run_audit_record(
    PIPELINE_RUN_ID, INPUT_TABLE, SOURCE_TABLE_VERSION,
    PIPELINE_STARTED_AT, pipeline_completed_at,
    n_raw, df_final_typed.count(), df_unresolved.count(), "SUCCESS",
)
write_run_audit(audit_df, AUDIT_TABLE)
print("\npipeline_run_audit row written:")
audit_df.show(truncate=False)


***
## Part 5: Quality Summary




### Task 5.1: Produce transformation summary metrics

Create a summary showing at minimum:

1. Total input rows
2. Successfully processed rows
3. Rows requiring exception handling
4. Rows successfully repaired
5. Rows still unresolved




In [ ]:
# ==== Task 5.1 execution: build_summary_metrics ====
total_input_rows = n_raw
requiring_exception_handling = n_exceptions_real  # from Part 3.1, pre-repair, on df_stage4
successfully_repaired = df_stage5.filter("record_status = 'repaired'").count()
still_unresolved = n_unresolved  # from Part 3.3
successfully_processed = n_success  # from Part 3.3 (clean + repaired)

summary_df = build_summary_metrics(
    total_input_rows, requiring_exception_handling, successfully_repaired,
    still_unresolved, successfully_processed,
)
summary_df.show(truncate=False)

print(f"\ncheck: total_input_rows > 0 -> {total_input_rows > 0} (sample file happens to have {total_input_rows})")
print(f"check: successfully_processed + still_unresolved == total_input_rows -> "
      f"{successfully_processed + still_unresolved == total_input_rows} "
      f"({successfully_processed} + {still_unresolved} == {total_input_rows})")
print(f"check: requiring_exception_handling == successfully_repaired + still_unresolved -> "
      f"{requiring_exception_handling == successfully_repaired + still_unresolved} "
      f"({requiring_exception_handling} == {successfully_repaired} + {still_unresolved})")

# df_stage5's row data is not needed again after this - the last stage still in
# memory once the notebook run completes.
df_stage5.unpersist()


---
**END OF CHALLENGE**

